# SVS Ensemble Run for Inverse Modeling

This script automates an ensemble run of the Soil Vegetation Snow (SVS) model to facilitate inverse modeling. This aids in identifying parameter ranges that best reproduce observed data.

## Key Features

* **Parameter Perturbation:** Systematically perturbs soil hydraulic parameters, including saturated water content (θs), saturated hydraulic conductivity (ksat), Clapp and Hornberg b-coefficient (bcoef), air entry pressure (psisat), and soil suction at field capacity (wfc_ψ).
* **Grid Search:**  Employs a grid search to explore parameter combinations efficiently.
* **Parallel Execution:** Leverages parallel processing to expedite the ensemble run, utilizing multiple cores for simultaneous model simulations.
* **Hourly Output:**  Generates hourly time-series output of key hydrological variables, such as evapotranspiration, overflow, drainage, soil moisture, and soil temperature.
* **Data Storage:**  Saves grid search configurations and parameter scenarios to CSV files for subsequent analysis.

## Methodology

1. **Initialization:**
   - Imports necessary libraries.
   - Defines file paths, soil properties, layering configurations, site parameters, and SVS model settings.
   - Establishes ranges and perturbation methods for soil hydraulic parameters.

2. **Grid Search:**
   - Creates a comprehensive dataframe (`df_grid_search`) representing all parameter combinations to be explored.
   - Calculates additional soil hydraulic properties (wfc, wwilt) based on the perturbed parameters.
   - Sets user-defined lower hydraulic boundary condition (trigger moisture) based on the last layer's saturated water content.

3. **Scenario Runs:**
   - Initializes an object (`perturb_inversed_1`) to manage the ensemble run.
   - Prepares a dataframe (`dfscens`) detailing parameter scenarios for each run.
   - Saves grid search and scenario dataframes to CSV files.
   - **(Commented out)** Initiates the parallel ensemble run, saving hourly output with specified variables.

## Parameter Perturbation Strategy

| Parameter                       | Min.    | Max.   | Perturbation Method | Unique Values |
| :-------------------------------- | :------ | :----- | :------------------ | :----------- |
| Soil Liquid Water Content at Saturation (θs)   | 0.33   | 0.43  | 0.02 increments    | 6           |
| Saturated Hydraulic Conductivity (ksat)       | 5.00e-07 | 5.01e-04 | logspace           | 10          |
| SWRC b-Coefficient (bcoef)                      | 0.2    | 5     | linspace           | 7           |
| SWRC Air Entry Pressure (psisat)               | 0.05   | 0.8   | linspace           | 5           |
| Soil Suction at Field Capacity (wfc_ψ)        | 1.02   | 3.36  | linspace           | 5           |


## Notes

- The script is designed to be run in a specific environment with pre-existing data and the SVS model executable.
- Ensure the `svspyed` package is installed for compatibility.
- Refer to SVS documentation for detailed explanations of model parameters.


## Author

Alireza Amani

## Date 

2024-07-17


In [1]:
# 1) Importing Libraries -------------------------------------------------------
from pathlib import Path
from collections import namedtuple
from copy import deepcopy
import pickle as pkl
import numpy as np
import pandas as pd

# these lines require you to have the `svspyed`` package installed
from svspyed.utils.helper_functions import (
    check_csv_columns, get_layering_df, populate_layering_df,
)
from svspyed.input_preparation.prep_svs import ModelInputData
from svspyed.model.svs_model import SVSModel
from svspyed.ensemble_run.perturb_run import PerturbAndRun

from helpers import bc_theta
# ______________________________________________________________________________
# 2) Variables -----------------------------------------------------------------

## 2.1. paths
paths = dict(
    meteo = Path("../Data/OriginalForcingData.csv"),
    svs_exec = Path("../SVS_Source_Code/svs_Jun12_2024"),
    working_dir = Path("./runs/"),
    default_met_file = Path("../Data/basin_forcing.met"),
    labranges = Path("../Data/LabParameterInfo_Mar2024.csv"),
)


## 2.2. SVS related variables
### 2.2.1. a container for the properties of a soil type
SoilType = namedtuple("SoilType", [
    'code',
    'sand', 'clay',
    'wsat', 'wfc', 'wwilt', 'wunfrz',
    'ksat',
    'psisat', 'bcoef',
])

### 2.2.2. a container for a soil cover
Enclosure = namedtuple("Enclosure", [
    'layering',
    'soil_layers'
])

### 2.2.3. a container for site-specific parameters required to configure SVS
SiteParams = namedtuple("SiteParams", [
    'deglat', 'deglng',
    'slop', 'zusl', 'ztsl',
    'observed_forcing', 'draindens', 'vf_type'
])

### 2.2.4. a container for SVS internal parameters
SVSParams = namedtuple("SVSParams", [
    'SCHMSOL', 'lsoil_freezing_svs1', 'soiltext', 'read_user_parameters',
    'save_minutes_csv', 'water_ponding', 'KFICE', 'OPT_SNOW', 'OPT_FRAC',
    'OPT_LIQWAT', 'WAT_REDIS', 'tperm', 'user_wfcdp',
    "OPT_VEGFLUX", "OPT_AGGFLUX", "OPT_VEGCOND",
    "user_VEGDAT_INDEX", "user_VEGDAT_VALUE",
    "user_LAM_VEGL_STAB", "user_LAM_VEGL_UNSTAB",

])

## 2.3. other variables
### columns to keep from the hourly SVS output
KEEPCOLS = (
  "date_utc", "member",
  'ACC_ET', 'ACC_OVFLW', 'ACC_DRAI',
) + tuple(
    f"WSOIL_{i}" for i in (3, 4, 8, 42, 43)
) + tuple(
    f"TPSOIL_{i}" for i in (3, 4, 8)
)

### number of cores to use for parallel runs
#### check the max. using `os.cpu_count()` or `multiprocessing.cpu_count()`
NCORES = 62

## 2.x. assertions
for key, value in paths.items():
    assert value.exists(), f"{key} does not exist."
# ______________________________________________________________________________
# 3) SVS default setup -------------------------------------------------------

## read the lab parameter ranges
df_labranges = pd.read_csv(paths["labranges"], index_col=0)

## 3.1. soil properties
TPsoil = SoilType(
    code = "Topsoil",
    sand = df_labranges.loc["SoilTexNaturalTerrainSand"]["Median"], # percent
    clay = df_labranges.loc["SoilTexNaturalTerrainClay"]["Median"], # percent
    wsat = 0.37, wfc = 0.10, wwilt = 0.032, wunfrz = 0.08, # volumetric fraction
    ksat = 1.416e-05, # m.s-1
    psisat = 0.24, bcoef = 3.6, # mH2O, unitless
)
CMsoil = SoilType(
    code = "CoverMaterial",
    sand = df_labranges.loc["SoilTexCoverMaterialSand"]["Median"], # percent
    clay = df_labranges.loc["SoilTexCoverMaterialClay"]["Median"], # percent
    wsat = 0.367, wfc = 0.10, wwilt = 0.01, wunfrz=0.04, # volumetric fraction
    ksat = 3.58e-06, # m.s-1
    psisat = 0.4, bcoef = 3.43, # mH2O, unitless
)

## 3.2. soil column
E1 = Enclosure(
    layering    = "15:2.5:Topsoil + 160:5:CoverMaterial + 15:2.5:CoverMaterial",
    soil_layers = [TPsoil, CMsoil],

    # Example: "190:5:CM" -> 190 cm total depth, divided into 5 cm increments, each with
    # the properties of CMsoil
    # could be changed to, for example, `15:2.5:CM + 175:5:CM` to have a 15 cm
    # top layer with 2.5 cm increments, and a 175 cm bottom layer with 5 cm
)

## 3.3. parameters describing the site, identical for all enclosures
site_params = SiteParams(
    deglat = 45.82, deglng = -72.37, # degrees latitude and longitude

    slop = 0.02, zusl = 10.0, ztsl = 1.5, # slope, heights for momentum and temp
    observed_forcing = "height", draindens = 0.0, vf_type = 13
    # drainage density, vegetation fraction type
)

## 3.4. parameters specific to the SVS model
### check SVS source code and doc for more info on these parameters
svs_params = SVSParams(
    SCHMSOL = "SVS", lsoil_freezing_svs1 = ".TRUE.", soiltext = "NIL",
    read_user_parameters = 1, save_minutes_csv = 0, water_ponding = 0,
    KFICE = 0, OPT_SNOW = 2, OPT_FRAC = 1, OPT_LIQWAT = 1, WAT_REDIS = 0,
    tperm = 280.15, user_wfcdp = 0.36,
    OPT_VEGFLUX = 3, OPT_AGGFLUX = 2, OPT_VEGCOND = 2,
    user_VEGDAT_INDEX = 13, user_VEGDAT_VALUE = 0.95,
    user_LAM_VEGL_STAB = 30.0, user_LAM_VEGL_UNSTAB = 4.4,
)

## 3.5. name of the directory containing the enclosure
HOST_DIR_NAME = "test_run"

## 3.6. mapping of the required meteo variables to the labels in the forcing file
## needed for the `save_csv_as_met` function
meteo_cols = {
    "utc_dtime": "datetime_utc",
    "air_temperature": "Air Temperature (degC)",
    "precipitation": "Precipitation (mm)",
    "wind_speed": "Wind Speed (m/s)",
    "atmospheric_pressure": "Atmospheric Pressure (Pa)",
    "shortwave_radiation": "Shortwave Radiation (W/m2)",
    "longwave_radiation": "Longwave Radiation (W/m2)",
    "specific_humidity": "Specific Humidity (kg/kg)",
    "relative_humidity": "Relative Humidity (%)"
}

### 3.6.1. check if the columns in the meteo file are correct
print("Checking meteo file columns...")
check_csv_columns(paths["meteo"], meteo_cols);

## 3.7. simulation start_date
START_DATE = "2017-182-04-00"
END_DATE = "2019-244-04-00"

## 3.8. spinup end date
SPINUP_END_DATE = "2018-07-01 00:00:00"

## 3.9. spinup date timezone
SPINUP_END_DATE_TZ = "America/Montreal"

## 3.10. name of the parameters to assign for each layer: all fields except `code`
layer_params = tuple(field for field in CMsoil._fields if field != 'code')

## 3.11. read layering information and populate the dataframe
dfenc = get_layering_df(E1)
dfenc = populate_layering_df(dfenc, E1, site_params)

## 3.12. initialize the input object
input_object = ModelInputData(
    work_dir_path=paths["working_dir"],
    soilcover_info=dfenc,
    metfile_path=None,
    exec_file_path=paths["svs_exec"],
    host_dir_name=HOST_DIR_NAME,
    meteo_col_names=meteo_cols,
    param_col_names={
        **{k: k for k in layer_params},
        **{k: k for k in site_params._fields},
    },
    model_params=svs_params._asdict(),
    start_date=START_DATE,
    end_date=END_DATE,
    spinup_end_date=SPINUP_END_DATE,
    time_zone=SPINUP_END_DATE_TZ,
    model_tsetp=1,
    copy_metfile=paths["default_met_file"],
)

## initialize the SVS model using the default setup
svs_model = SVSModel(input_object, remove_old_host_folder=True, verbose=True)

# ______________________________________________________________________________
# 4) Grid search for the first set of parameters -------------------------------


## 4.2.
perturb_values = {
    "wsat": np.arange(0.33, 0.44, 0.02).round(2),
    "ksat": np.logspace(-6.3, -3.3, 10).round(8),
    "bcoef": np.linspace(0.2, 5, 7).round(1),
    "psisat": np.linspace(0.05, 0.8, 5).round(2),
    "wfc_psi": np.linspace(10/9.81, 33/9.81, 5).round(2),
}

## print total number of runs
print(f"Total number of runs: {np.prod([len(v) for v in perturb_values.values()])}")

## make 4d grid search dataframe, each row corr to one scenario (one run)
df_grid_search = pd.DataFrame({
    "wsat": [], "ksat": [], "bcoef": [], "psisat": [], "wfc": [], "run_id": []
})

### populate the dataframe
for i, wsat in enumerate(perturb_values["wsat"]):
    for j, ksat in enumerate(perturb_values["ksat"]):
        for k, bcoef in enumerate(perturb_values["bcoef"]):
            for l, psisat in enumerate(perturb_values["psisat"]):
                for m, wfc_psi in enumerate(perturb_values["wfc_psi"]):
                    df_grid_search = pd.concat([
                        df_grid_search,
                        pd.DataFrame({
                            "wsat": wsat,
                            "ksat": ksat,
                            "bcoef": bcoef,
                            "psisat": psisat,
                            "wfc_psi": wfc_psi,
                            "run_id": f"{i:02d}{j:02d}{k:02d}{l:02d}{m:02d}"
                        }, index=[0])
                    ])
# ______________________________________________________________________________
# 5) scenario runs -------------------------------------------------------------

## create a dict
inversed_1 = dict()

### number of layers
nalyers = len(dfenc)

for i, row in df_grid_search.iterrows():
    run_id = row["run_id"]

    inversed_1[run_id] = {
        "wsat": np.full(nalyers, row["wsat"]),
        "ksat": np.full(nalyers, row["ksat"]),
        "bcoef": np.full(nalyers, row["bcoef"]),
        "psisat": np.full(nalyers, row["psisat"]),
        "wfc": np.full(nalyers, np.nan),
        "wwilt": np.full(nalyers, np.nan),
    }

    for layer in range(nalyers):
        inversed_1[run_id]["wfc"][layer] = bc_theta(
            psi_ae=inversed_1[run_id]["psisat"][layer],
            psi_m2ho=row["wfc_psi"],
            theta_s=inversed_1[run_id]["wsat"][layer],
            b=inversed_1[run_id]["bcoef"][layer]
        ).round(3)

        inversed_1[run_id]["wwilt"][layer] = bc_theta(
            psi_ae=inversed_1[run_id]["psisat"][layer],
            psi_m2ho=1500 / 9.81,
            theta_s=inversed_1[run_id]["wsat"][layer],
            b=inversed_1[run_id]["bcoef"][layer]
        ).round(3)

        # if wfc is not > wwilt, set wfc wwilt + 0.01
        if inversed_1[run_id]["wfc"][layer] <= inversed_1[run_id]["wwilt"][layer]:
            inversed_1[run_id]["wfc"][layer] = inversed_1[run_id]["wwilt"][layer] + 0.01

    #### set user_wfcdp to 0.99 of last layer wsat
    inversed_1[run_id]["user_wfcdp"] = inversed_1[run_id]["wsat"][-1] * 0.99


perturb_inversed_1 = PerturbAndRun(
    deepcopy(input_object), inversed_1,
    met_paths=[paths["default_met_file"]], njobs=NCORES,
    verbose=True,
)

perturb_inversed_1.create_param_scen_df()
dfscens = perturb_inversed_1.dfscenarios.copy()


## save the dataframes
df_grid_search.to_csv(
    "./Results/dfGridSearch_5Params_Mar8.csv", index=True, index_label="run_id",
)

dfscens.to_csv(
    "./Results/param_scenarios_5Params_Mar8.csv", index=True, index_label="run_id",
)

## let the ensemble run begin (uncomment to run)
# perturb_inversed_1.run_all_parallel(
#     output_time_scale="hourly", effort_id="InversedModelling_5Params_Mar8_",
#     keepcols=KEEPCOLS
# )
# ______________________________________________________________________________

Checking meteo file columns...
All columns found!

a folder named `test_run` is created inside the dir: `runs`


The `MESH_input_soil_levels.txt` file is created inside the host folder located at 
dir: `runs/test_run`.
In this file 44 soil layers are defined.

.met file copied to: runs/test_run/basin_forcing.met

The MESH_input_run_options.ini is created inside the host folder
at: `runs/test_run`
Simulatuin start time (UTC): 2017-07-01 04:00:00
Simulatuin end time (UTC): 2019-09-01 04:00:00

MESH_parameters.txt created in the host folder at: `runs/test_run`

The SVS executable file is copied into the host folder. named `SVS_exe`.

SVSModel instance created.

Total number of runs: 10500

Number of scenarios: 10500



# Soil Volumetric Liquid Water Content (VWC) Extraction for Inverse Modeling

This script extracts simulated soil VWC at specific depths (75mm and 1850mm) from an ensemble of SVS model runs. The extracted data is then saved as Parquet files for further analysis, particularly for use in inverse modeling procedures to calibrate model parameters by comparing simulations against field measurements.

## Data Source

The script expects the output files (in Feather format) from the ensemble SVS model run to be located in the following directory:

``` 
/Users/xxx/xxx/xxx/ens_checkpoint_5params
``` 

**Important Note:** Replace the `xxx` placeholders with your actual file path.

## Extraction Function

A single, versatile function, `read_wsoil_depth(file, depth_columns)`, is defined to handle the extraction process for both depths:

1.  **Reads Data:** Reads the specified Feather file (`file`) into a Pandas DataFrame.
2.  **Calculates Mean VWC:** Calculates the mean VWC at the desired depth using the provided column names (`depth_columns`). The new column is named based on the provided depth column names. 
3.  **Filters Time Period:** Selects data from April 1, 2019, to September 1, 2019.
4.  **Returns DataFrame:** Returns a DataFrame with the `member` column (identifying the ensemble member) and the calculated mean VWC column.

## Extraction and Saving Process

The script performs the following steps:

1.  **Collects Files:**  Identifies all Feather files (`*.feather`) within the `output_dir`.
2.  **Extracts 75mm Data:**
    *   Iterates through each file and applies the `read_wsoil_depth` function with `["WSOIL_3", "WSOIL_4"]` to extract mean VWC at 75mm.
    *   Concatenates the extracted data from all files.
    *   Saves the combined data as `WSOIL75mm_5Params_Mar8.parquet`.
3.  **Extracts 1850mm Data:** Repeats the same process as step 2, but uses `["WSOIL_42", "WSOIL_43"]` to extract mean VWC at 1850mm and saves the data as `WSOIL1850mm_5Params_Mar8.parquet`.


In [18]:
from pathlib import Path
import pandas as pd

# Data Source
output_dir = Path("/Users/pinovanrijn/Desktop/no_sync/ens_checkpoint_5params")

def read_wsoil_depth(file, depth_columns):
    """Reads a Feather file and extracts mean soil VWC at a specified depth."""

    df = pd.read_feather(file)
    df[f"{depth_columns[0]}_{depth_columns[1]}"] = df[depth_columns].mean(axis=1)
    df.set_index("date_utc", inplace=True)
    return df[["member", f"{depth_columns[0]}_{depth_columns[1]}"]].loc["2019-04-01":"2019-09-01"]

# Extract and Save
list_files = output_dir.glob("*.feather")
_ = pd.concat(
    [read_wsoil_depth(file, ["WSOIL_3", "WSOIL_4"]) for file in list_files], axis=0
).to_parquet("./Results/WSOIL75mm_5Params_Mar8.parquet")

list_files = output_dir.glob("*.feather")
_ = pd.concat(
    [read_wsoil_depth(file, ["WSOIL_42", "WSOIL_43"]) for file in list_files], axis=0
).to_parquet("./Results/WSOIL1850mm_5Params_Mar8.parquet")